# 🧭 Mission 1: Locate and Retrieve the Dataset

```{admonition} Your mission
:class: tip

Your investigation begins in **ioerDATA**.

In this mission, you will:

- 🔍 locate a research dataset
- 🧾 identify its persistent identifier (PID)
- 📦 retrieve metadata through the Dataverse API
- 🔐 recognise public and restricted files
- ⬇️ download data reproducibly

🏅 **Badge to unlock:** Data Collector

```{raw} html
<div id="first-visit-message"></div>

<script>
const visitedKey = "urban_data_mission_1_visited";

if (!localStorage.getItem(visitedKey)) {

  localStorage.setItem(visitedKey, "true");

  document.getElementById("first-visit-message").innerHTML = `
    <div class="admonition note">
      <p class="admonition-title">🎮 New Mission Started</p>
      <p>
        Welcome, Urban Data Investigator.
        This is your first time entering Mission 1.
      </p>
    </div>
  `;

} else {

  document.getElementById("first-visit-message").innerHTML = `
    <div class="admonition tip">
      <p class="admonition-title">🔁 Mission Revisited</p>
      <p>
        You have already started this mission.
        Continue your investigation.
      </p>
    </div>
  `;
}
</script>
```

# 📁 Your case file

We will work with the replication package:

>**Localized assessment of urban forest structures with 3D indicators**

Think of the dataset as your **case file**. Before analysing anything, we first need to find it and understand how its files can be accessed.

# 🔍 Step 1: Locate the dataset

>Instead of manually browsing and downloading files, we can ask the **Dataverse API** to find the dataset for us.

We begin with its title.

In [1]:
import requests

base_url = "https://data.fdz.ioer.de"
search_url = f"{base_url}/api/search"

query = "Localized assessment of urban forest structures with 3D indicators"

params = {
    "q": query,
    "type": "dataset",
    "per_page": 10
}

response = requests.get(
    search_url,
    params=params,
    timeout=30
)

response.raise_for_status()

items = response.json().get("data", {}).get("items", [])

if not items:
    raise ValueError(f"No dataset found for: {query}")

dataset = items[0]

persistent_id = dataset.get("global_id")

print("Dataset:", dataset.get("name"))
print("PID:", persistent_id)

Dataset: Replication package for: Localized assessment of urban forest structures with 3D indicators
PID: doi:10.71830/CDAXYF


# 🧾 Follow the PID

The search returns:

**`doi:10.71830/CDAXYF`**

This persistent identifier gives us a stable way to refer to the dataset.

That is much more reproducible than relying on a manually copied download link.

```{admonition} 🔍 Evidence check
:class: note

Check the result above.

- Does the title match our case file?
- Was a PID returned?

If both are present, we have located the correct dataset.


# 📦 Step 2: Retrieve the metadata

Before downloading files, let's inspect the dataset metadata.

Metadata tells us what the package contains and which files we can access.

In [2]:
import json

dataset_url = f"{base_url}/api/datasets/:persistentId/"

response = requests.get(
    dataset_url,
    params={"persistentId": persistent_id},
    timeout=30
)

response.raise_for_status()

dataset_metadata = response.json()

with open(
    "g_dataset_metadata.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        dataset_metadata,
        file,
        indent=2,
        ensure_ascii=False
    )

print("Metadata retrieved and saved.")

Metadata retrieved and saved.


# 🧾 Why save the metadata?

>The metadata records important context about the dataset, including its files and access conditions.

Saving it alongside the analysis also helps document **exactly what was retrieved**.

# 🔓 Step 3: Download the open files

The package contains both **open** and **restricted** files.

Our first download should respect those access conditions automatically:

> **Download what is openly available and record what is restricted.**

In [5]:
import os

output_folder = "data/g_raw"
os.makedirs(output_folder, exist_ok=True)

files = (
    dataset_metadata.get("data", {})
    .get("latestVersion", {})
    .get("files", [])
)

if not files:
    raise ValueError("No files found in the dataset metadata.")

downloaded = []
skipped = []

for item in files:

    data_file = item.get("dataFile", {})

    file_id = data_file.get("id")
    filename = data_file.get("filename", f"file_{file_id}")
    filesize = data_file.get("filesize")

    if item.get("restricted", True):
        skipped.append(
            (filename, filesize, "restricted")
        )
        continue

    if not file_id:
        skipped.append(
            (filename, filesize, "missing file ID")
        )
        continue

    download_url = (
        f"{base_url}/api/access/datafile/{file_id}"
    )

    try:

        with requests.get(
            download_url,
            stream=True,
            timeout=60
        ) as response:

            response.raise_for_status()

            local_path = os.path.join(
                output_folder,
                filename
            )

            total_bytes = 0

            with open(local_path, "wb") as file:

                for chunk in response.iter_content(
                    chunk_size=1024 * 1024
                ):

                    if chunk:
                        file.write(chunk)
                        total_bytes += len(chunk)

        downloaded.append(
            (filename, total_bytes)
        )

    except requests.exceptions.RequestException as error:

        skipped.append(
            (filename, filesize, str(error))
        )

In [6]:
print("Downloaded:")
for name, size in downloaded:
    print(f" ✓ {name} ({size:,} bytes)")

print("\nNot downloaded:")
for name, size, reason in skipped:
    print(f" 🔒 {name} — {reason}")

Downloaded:
 ✓ README.md (11,959 bytes)
 ✓ lcz_2018_classcodes.csv (528 bytes)
 ✓ localized_assessment_of_urban_forest_structures_with_3d_indicators_replication_static.html (13,596,982 bytes)
 ✓ localized_assessment_of_urban_forest_structures_with_3d_indicators_replication_workflow.ipynb (99,742 bytes)
 ✓ urban_forest_3d_indicators_graphical_abstract.png (202,373 bytes)

Not downloaded:
 🔒 amsterdam_3D_canopy_stats_lczv3_grid100m_lau2021_epsg28992.gpkg — restricted
 🔒 amsterdam_gwr_canopy_indicators_adaptive_bandwidth_12_bisquare_kernel.rds — restricted
 🔒 berlin_3D_canopy_stats_lczv3_grid100m_urau2021_epsg25833.gpkg — restricted
 🔒 berlin_gwr_canopy_indicators_adaptive_bandwidth_12_bisquare_kernel.rds — restricted


```{admonition} 🕵️ Investigator challenge
:class: note

Look at the two groups of files.

🔓 Which files were downloaded automatically?  
🔒 Which files require additional access?

Notice that our code did **not try to bypass the restrictions**. It recognised them and documented them instead.

# 🔐 Step 4: Access authorised restricted files

Some research files require permission.

>If your ioerDATA account has access, you can authenticate using a **personal API token**.

```{warning}
Treat an API token like a password.

Never publish it in a notebook, share it in screenshots, or commit it to Git.

In [7]:
from getpass import getpass
from pathlib import Path

output_folder = Path("data/g_raw")
output_folder.mkdir(
    parents=True,
    exist_ok=True
)

api_token = getpass(
    "Paste your ioerDATA API token: "
)

headers = {
    "X-Dataverse-key": api_token
}

Paste your ioerDATA API token:  ········


# 🛠️ Build a reusable download function

Instead of repeating the same download code for every restricted file, we define it once and reuse it.

This keeps the workflow shorter and easier to maintain.

In [15]:
def download_file(file_id, filename, headers):
    url = f"{base_url}/api/access/datafile/{file_id}"
    path = output_folder / filename

    with requests.get(url, headers=headers, stream=True, timeout=60) as response:
        if response.status_code in (401, 403):
            raise PermissionError(
                f"Access denied for {filename}. "
                "Check that your token is valid and that your account has file access."
            )

        response.raise_for_status()

        total = int(response.headers.get("content-length", 0))

        with open(path, "wb") as file, tqdm(
            total=total,
            unit="B",
            unit_scale=True,
            desc=filename,
        ) as progress:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    file.write(chunk)
                    progress.update(len(chunk))

    return path.stat().st_size

# ⬇️ Step 5: Retrieve the restricted files

Now we use the same metadata to identify restricted files.

Only files that your account is authorised to access will be downloaded.

In [16]:
for item in files:
    if item.get("restricted", False):
        data_file = item["dataFile"]

        size = download_file(
            file_id=data_file["id"],
            filename=data_file["filename"],
            headers=headers
        )

        print(f"Downloaded {data_file['filename']} ({size:,} bytes)")

amsterdam_3D_canopy_stats_lczv3_grid100m_lau2021_epsg28992.gpkg: 0.00B [00:00, ?B/s]

Downloaded amsterdam_3D_canopy_stats_lczv3_grid100m_lau2021_epsg28992.gpkg (6,025,216 bytes)


amsterdam_gwr_canopy_indicators_adaptive_bandwidth_12_bisquare_kernel.rds: 0.00B [00:00, ?B/s]

Downloaded amsterdam_gwr_canopy_indicators_adaptive_bandwidth_12_bisquare_kernel.rds (2,447,010 bytes)


berlin_3D_canopy_stats_lczv3_grid100m_urau2021_epsg25833.gpkg: 0.00B [00:00, ?B/s]

Downloaded berlin_3D_canopy_stats_lczv3_grid100m_urau2021_epsg25833.gpkg (25,026,560 bytes)


berlin_gwr_canopy_indicators_adaptive_bandwidth_12_bisquare_kernel.rds: 0.00B [00:00, ?B/s]

Downloaded berlin_gwr_canopy_indicators_adaptive_bandwidth_12_bisquare_kernel.rds (13,396,251 bytes)


# 🔍 What did we retrieve?

The replication package now gives us several different kinds of evidence:

>- 📄 documentation
>- 💻 analysis code
>- 🗺️ spatial GeoPackages
>- 📊 analysis outputs
>- 🖼️ supporting visual material

We have not analysed them yet.

That is the job of the next mission.

```{tip}
A reproducible download workflow records more than the files themselves.

It also records:

**where the data came from → how it was identified → which files were accessible → how they were retrieved**

```{raw} html
<div class="admonition tip">
  <p class="admonition-title">🎉 Mission Complete — Badge Unlocked!</p>

  <h3>🏆 Data Collector</h3>

  <p>
    You successfully located and retrieved your research case file.
  </p>

  <ul>
    <li>✔ Dataset located</li>
    <li>✔ Persistent identifier found</li>
    <li>✔ Metadata retrieved</li>
    <li>✔ Access restrictions recognised</li>
    <li>✔ Authorised files downloaded</li>
  </ul>

  <p><strong>Status:</strong> Mission 1 complete</p>

  <button
    onclick="completeMission1()"
    style="
      padding: 10px 16px;
      font-weight: bold;
      border-radius: 6px;
      cursor: pointer;
    ">
    ✅ Complete Mission 1
  </button>

  <p
    id="mission-1-complete-message"
    style="
      font-weight: bold;
      margin-top: 12px;
    ">
  </p>
</div>

<script>
function completeMission1() {

  localStorage.setItem(
    "urban_data_mission_1_completed",
    "true"
  );

  localStorage.setItem(
    "urban_data_badge_1",
    "Data Collector"
  );

  localStorage.setItem(
    "urban_data_mission_1_completed_at",
    new Date().toISOString()
  );

  document.getElementById(
    "mission-1-complete-message"
  ).innerHTML =
    "🔓 Mission 2 unlocked! Badge added to your gallery.";
}
</script>

```{admonition} 🎮 Mission Progress
:class: note, dropdown

**Your Journey**

🟢 **Mission 1: Locate & Retrieve Data** ← *Current*  
⚪ Mission 2: Inspect the Package 🔒  
⚪ Mission 3: Understand the Data 🔒  
⚪ Mission 4: Generate Insights 🔒  
⚪ Mission 5: Create Visual Evidence 🔒  
⚪ Mission 6: Map the City 🔒  
⚪ Mission 7: Extend the Analysis 🔒  
⚪ Final Mission: Tell the Data Story 🔒  

**Progress:** ▰▱▱▱▱▱▱▱ 1/8 Complete
```

```{admonition} 🚧 Next Mission Preview
:class: note, dropdown

# 📦 Mission 2: Inspect the Package

The files have arrived. Now you need to find out **what is inside them**.

You will learn how to:

- 📁 explore the replication-package structure
- 📄 identify documentation and workflow files
- 🗺️ locate the main spatial datasets
- 🔎 decide which files matter for the investigation

**Your next challenge: inspect the evidence package.**